# Bangalore Tech Salary Decoder
**The Unlox Academy - 2-Hour Live Project**

Cleaning and analysing 1,000 Bengaluru tech salaries to find what really drives pay:
role, experience, company type, skills and education.

## Task 1 - Load and Inspect

In [11]:
import pandas as pd
import numpy as np

# load the dataset
df = pd.read_csv("bangalore_tech_salaries.csv")

print("First 5 rows:\n", df.head())

print("\nShape of Dataset:\n", df.shape)

print("\nDataset Information:")
df.info()

print("\nData types:\n", df.dtypes)

print("\nMissing values:\n", df.isnull().sum())

print("\nDuplicated rows:\n", df.duplicated().sum())


print('-' * 25, "Dataset Summary", '-' * 25)
print("Rows :", df.shape[0])
print("Columns :", df.shape[1])
print("Duplicate Rows :", df.duplicated().sum())

missing_values = df.isnull().sum()
print("\nColumns with Missing Values")
print(missing_values[missing_values > 0])

# short written summary of dataset quality (facts only)
print('-' * 25, "Quality Summary", '-' * 25)
print("- Dataset has", df.shape[0], "rows and", df.shape[1], "columns.")
print("-", df.duplicated().sum(), "rows are exact duplicates and should be removed.")
print("- Previous_CTC is missing for", int(missing_values["Previous_CTC"]), "rows (these are freshers).")
print("- Current_CTC and Previous_CTC are stored as text, not numbers.")
print("- Role, company_TYPE, Education_Tier have many spelling variants.")
print("- Data needs cleaning before analysis.")

First 5 rows:
   Employee_ID         Role   years_exp Current_CTC Previous_CTC       Company  \
0     BLR0065            DS          6        49.4     32.4 LPA           Ola   
1     BLR0080        DevOps          0         9.7          NaN  Walmart Labs   
2     BLR0166   SDE Backend          6   3,360,000     24.2 LPA       Postman   
3     BLR0219  SDE Frontend          8    41.4 LPA         30.3  Amazon India   
4     BLR0491  Product Lead          5   3,640,000     30.6 LPA   QuickPay AI   

  company_TYPE                                             Skills  \
0      Unicorn  JIRA, JavaScript, Spring Boot, Figma, NumPy, A...   
1          MNC     Tableau, Python, Figma, JavaScript, TensorFlow   
2     Mid-size               Agile, Excel, GCP, MongoDB, Java, ML   
3          MNC                      PyTorch, Redis, System Design   
4  early-stage                      Tableau, Deep Learning, Agile   

          Location Education_Tier  Joining_Year         Work_Mode  
0       HSR Lay

## Task 2 - Clean

In [12]:
# ---- Task 2: clean the data ----

# rename columns to snake_case (lowercase, underscores instead of spaces)
df.columns = df.columns.str.strip().str.lower()
df.columns = df.columns.str.replace(" ", "_", regex=False)

# drop duplicate rows
df = df.drop_duplicates()
print("Rows after removing duplicates :", len(df))

# ---- convert Role variants to clean names ----
# I first checked df["role"].unique() and saw ~40 spellings.
# Many mean the same job, so I built this mapping by hand.
# AI-assisted: asked for help grouping the role variants I found in the data.
role_map = {
    "BA": "Business Analyst", "Business Analyst": "Business Analyst", "Business Systems Analyst": "Business Analyst",
    "Data Scientist": "Data Scientist", "DS": "Data Scientist", "Data Science Engineer": "Data Scientist",
    "DA": "Data Analyst", "Data Analyst": "Data Analyst",
    "BI Analyst": "BI Analyst", "Analytics Engineer": "Analytics Engineer",
    "ML Engineer": "ML Engineer",
    "DevOps": "DevOps Engineer", "DevOps Engineer": "DevOps Engineer",
    "Site Reliability Engineer": "Site Reliability Engineer", "SRE": "Site Reliability Engineer", "Infra Engineer": "Site Reliability Engineer",
    "SDE Backend": "SDE Backend", "SDE-Backend": "SDE Backend", "Backend Dev": "SDE Backend", "BE": "SDE Backend", "Backend Engineer": "SDE Backend", "Backend Developer": "SDE Backend",
    "SDE Frontend": "SDE Frontend", "SDE-Frontend": "SDE Frontend", "Frontend Developer": "SDE Frontend", "Frontend Engineer": "SDE Frontend", "Frontend Dev": "SDE Frontend", "FE": "SDE Frontend",
    "SDE FS": "SDE Full-Stack", "Full Stack Engineer": "SDE Full-Stack", "FullStack": "SDE Full-Stack", "SDE Full-Stack": "SDE Full-Stack", "Fullstack Dev": "SDE Full-Stack",
    "Product Manager": "Product Manager", "PM": "Product Manager", "Product Lead": "Product Manager", "Sr PM": "Product Manager",
    "UI/UX": "UI/UX Designer", "UI Designer": "UI/UX Designer", "Product Designer": "UI/UX Designer", "UX Designer": "UI/UX Designer", "Designer": "UI/UX Designer",
}
df["role_clean"] = df["role"].str.strip().map(role_map)
print("Roles not mapped (should be 0) :", df["role_clean"].isna().sum())

# ---- convert CTC strings to numbers (in LPA) ----
# Current_CTC has 4 formats: '15.5 LPA', '15.5', '15,50,000', 'Rs 15.5 LPA'
# If there is a comma it is in rupees, so divide by 100000.
# AI-assisted: asked for a parser that handles all 4 formats correctly.
def convert_ctc(value):
    if pd.isna(value):
        return np.nan
    value = str(value).strip()
    if "," in value:
        value = value.replace(",", "")
        value = value.replace("\u20b9", "").replace("LPA", "").strip()
        return float(value) / 100000
    value = value.replace("\u20b9", "").replace("LPA", "").strip()
    return float(value)

df["ctc_lpa"] = df["current_ctc"].apply(convert_ctc)
df["prev_ctc_lpa"] = df["previous_ctc"].apply(convert_ctc)
print("Current CTC range :", df["ctc_lpa"].min(), "to", df["ctc_lpa"].max())
print("Previous CTC missing (freshers) :", df["prev_ctc_lpa"].isna().sum())

# ---- standardise company_type ----
company_map = {
    "MNC": "MNC", "mnc": "MNC",
    "Mid-size": "Mid-size", "mid-size": "Mid-size", "MID-SIZE": "Mid-size",
    "Unicorn": "Unicorn", "unicorn": "Unicorn", "UNICORN": "Unicorn",
    "Early-stage": "Early-stage", "early-stage": "Early-stage", "EARLY-STAGE": "Early-stage",
}
df["company_type_clean"] = df["company_type"].str.strip().map(company_map)

# ---- standardise education tier ----
edu_map = {
    "Tier 1": "Tier 1", "Tier-1": "Tier 1", "T1": "Tier 1", "1": "Tier 1",
    "Tier 2": "Tier 2", "Tier-2": "Tier 2", "T2": "Tier 2", "2": "Tier 2",
    "Tier 3": "Tier 3", "Tier-3": "Tier 3", "T3": "Tier 3", "3": "Tier 3",
}
df["education_tier_clean"] = df["education_tier"].astype(str).str.strip().map(edu_map)

# ---- standardise work_mode ----
work_map = {
    "Remote": "Remote", "WFH": "Remote",
    "Hybrid": "Hybrid", "Hybrid (3 days)": "Hybrid",
    "Onsite": "Onsite", "Work from Office": "Onsite",
}
df["work_mode_clean"] = df["work_mode"].str.strip().map(work_map)

# ---- clean skills (fill missing with empty, make lowercase) ----
df["skills_clean"] = df["skills"].fillna("").str.strip().str.lower()

# ---- create experience bands: 0-1, 2-3, 4-5, 6+ ----
bins = [-1, 1, 3, 5, 100]
labels = ["0-1", "2-3", "4-5", "6+"]
df["exp_band"] = pd.cut(df["years_exp"], bins=bins, labels=labels).astype(str)

# ---- verify the cleaning (dtypes + value_counts) ----
print('-' * 25, "After Cleaning", '-' * 25)
print("Shape :", df.shape)
print("\nData types:\n", df.dtypes)
print("\nRole counts:\n", df["role_clean"].value_counts())
print("\nCompany type counts:\n", df["company_type_clean"].value_counts())
print("\nEducation tier counts:\n", df["education_tier_clean"].value_counts())

Rows after removing duplicates : 1000
Roles not mapped (should be 0) : 0
Current CTC range : 5.2 to 84.4
Previous CTC missing (freshers) : 194
------------------------- After Cleaning -------------------------
Shape : (1000, 20)

Data types:
 employee_id              object
role                     object
years_exp                 int64
current_ctc              object
previous_ctc             object
company                  object
company_type             object
skills                   object
location                 object
education_tier           object
joining_year              int64
work_mode                object
role_clean               object
ctc_lpa                 float64
prev_ctc_lpa            float64
company_type_clean       object
education_tier_clean     object
work_mode_clean          object
skills_clean             object
exp_band                 object
dtype: object

Role counts:
 role_clean
SDE Backend                  114
UI/UX Designer               113
Product Man

## Task 3 - Five Business Questions

Each question has a comment explaining the approach, the code, the printed result,
then a markdown insight written from the real numbers.

### Q3.1 - CTC distribution by role

In [13]:
# Q3.1: find median, mean, min, max CTC for each role
# Approach: group by role_clean and build a small summary table, sort by median.
grouped = df.groupby("role_clean")["ctc_lpa"]

result = pd.DataFrame()
result["median"] = grouped.median()
result["mean"] = grouped.mean()
result["min"] = grouped.min()
result["max"] = grouped.max()

result = result.sort_values("median", ascending=False)
print('-' * 25, "Q3.1: CTC by Role", '-' * 25)
print(result.round(1))

------------------------- Q3.1: CTC by Role -------------------------
                           median  mean   min   max
role_clean                                         
Product Manager              31.3  34.1  10.8  80.1
ML Engineer                  27.5  30.3  10.0  64.5
Data Scientist               24.2  26.7  10.9  75.6
Site Reliability Engineer    23.3  23.5   8.8  55.0
SDE Full-Stack               22.4  25.4   8.9  71.7
Analytics Engineer           21.4  20.9   6.8  37.9
SDE Frontend                 21.2  22.2   6.7  84.4
SDE Backend                  21.1  23.3   7.9  55.1
Business Analyst             19.9  21.9   6.8  52.7
DevOps Engineer              19.4  22.9   9.7  60.3
UI/UX Designer               18.9  21.0   6.2  63.3
BI Analyst                   16.5  16.8   5.2  36.3
Data Analyst                 16.3  17.1   5.6  43.4


**Insight (Q3.1):** Product Managers earn the highest median CTC at **31.3 LPA**,
almost 2x of Data Analysts, the lowest at **16.3 LPA**. DevOps Engineers show a big
mean-vs-median gap (mean 22.9 vs median 19.4) - a sign a few high earners pull the
average up, so the **median** is the safer number to read here.

### Q3.2 - Experience curve for SDE Backend

In [14]:
# Q3.2: bin SDE Backend people by experience and find median CTC per band
# Approach: filter to SDE Backend, group by exp_band, find medians,
# then find the % growth from one band to the next.
sde_backend = df[df["role_clean"] == "SDE Backend"]
band_ctc = sde_backend.groupby("exp_band")["ctc_lpa"].median().round(1)

print('-' * 25, "Q3.2: SDE Backend by Experience", '-' * 25)
prev = None
for band in band_ctc.index:
    value = band_ctc[band]
    if prev is None:
        print(band, "years :", value, "LPA median")
    else:
        growth = (value - prev) / prev * 100
        print(band, "years :", value, "LPA median   (+" + str(round(growth)) + "% growth)")
    prev = value

------------------------- Q3.2: SDE Backend by Experience -------------------------
0-1 years : 11.6 LPA median
2-3 years : 20.0 LPA median   (+72% growth)
4-5 years : 25.8 LPA median   (+29% growth)
6+ years : 40.4 LPA median   (+57% growth)


**Insight (Q3.2):** SDE Backend pay grows from **11.6 LPA** (0-1 yrs) to
**40.4 LPA** (6+ yrs) - about **3.5x** overall. The biggest single jump is the move
from 0-1 to 2-3 years (**+72%**), worth more in percentage terms than any later band.

### Q3.3 - Skill premium for SDEs

In [15]:
# Q3.3: for each skill, compare median CTC of SDEs who have it vs who don't
# Approach: keep only SDE roles. For each skill, split into has / does-not-have
# using str.contains (regex=False), then compare medians.
sde = df[(df["role_clean"] == "SDE Backend") |
         (df["role_clean"] == "SDE Frontend") |
         (df["role_clean"] == "SDE Full-Stack")]

skills = ["AWS", "ML", "System Design", "Kubernetes"]

skill_names = []
with_ctc_list = []
without_ctc_list = []
premium_list = []

for skill in skills:
    has_skill = sde["skills_clean"].str.contains(skill.lower(), regex=False)
    with_ctc = sde[has_skill]["ctc_lpa"].median()
    without_ctc = sde[~has_skill]["ctc_lpa"].median()
    premium = (with_ctc - without_ctc) / without_ctc * 100

    skill_names.append(skill)
    with_ctc_list.append(round(with_ctc, 1))
    without_ctc_list.append(round(without_ctc, 1))
    premium_list.append(round(premium, 1))

skill_table = pd.DataFrame({
    "skill": skill_names,
    "with_skill": with_ctc_list,
    "without": without_ctc_list,
    "premium_pct": premium_list,
})
skill_table = skill_table.sort_values("premium_pct", ascending=False)

print('-' * 25, "Q3.3: Skill Premium for SDEs", '-' * 25)
print(skill_table.to_string(index=False))

------------------------- Q3.3: Skill Premium for SDEs -------------------------
        skill  with_skill  without  premium_pct
System Design        25.3     21.0         20.5
           ML        23.9     21.3         12.4
   Kubernetes        23.2     21.5          7.9
          AWS        20.9     21.9         -4.3


**Insight (Q3.3):** **System Design** gives the biggest premium of the skills
tested (**+20.5%**, 25.3 vs 21.0 LPA). ML (**+12.4%**) and Kubernetes (**+7.9%**) also
help. AWS shows a small *discount* (**-4.3%**) - it has become a basic expected skill,
not a differentiator.

### Q3.4 - Company-type premium, same role

In [16]:
# Q3.4: compare median CTC across company types for ONE role (SDE Backend)
# Approach: within SDE Backend only, group by company_type_clean and find medians.
sde_backend = df[df["role_clean"] == "SDE Backend"]
company_ctc = sde_backend.groupby("company_type_clean")["ctc_lpa"].median()
company_ctc = company_ctc.sort_values(ascending=False)

unicorn_ctc = company_ctc["Unicorn"]
mnc_ctc = company_ctc["MNC"]
premium = (unicorn_ctc - mnc_ctc) / mnc_ctc * 100

print('-' * 25, "Q3.4: Company Type Premium (SDE Backend)", '-' * 25)
print(company_ctc.round(1))
print("Unicorn vs MNC premium :", round(premium, 1), "%")

------------------------- Q3.4: Company Type Premium (SDE Backend) -------------------------
company_type_clean
Unicorn        27.4
MNC            20.5
Mid-size       19.5
Early-stage    18.6
Name: ctc_lpa, dtype: float64
Unicorn vs MNC premium : 33.7 %


**Insight (Q3.4):** For the exact same SDE Backend role, Unicorns pay a
**27.4 LPA** median vs **20.5 LPA** at MNCs - a **33.7%** premium, and about **47%**
more than Early-stage (18.6 LPA). Company type moves pay almost as much as a few
years of experience.

### Q3.5 - Underpaid professionals

In [17]:
# Q3.5: find the most underpaid people vs their peer group
# Approach:
# 1. find median CTC for each (role, company_type, exp_band) group
# 2. gap = own CTC - group median
# 3. keep only groups with 10 or more people, so the median is reliable
# 4. sort by gap and take the 10 most negative
df["group_median"] = df.groupby(["role_clean", "company_type_clean", "exp_band"])["ctc_lpa"].transform("median")
df["gap"] = df["ctc_lpa"] - df["group_median"]
df["group_size"] = df.groupby(["role_clean", "company_type_clean", "exp_band"])["ctc_lpa"].transform("size")

reliable = df[df["group_size"] >= 10]
underpaid = reliable.sort_values("gap").head(10)

print('-' * 25, "Q3.5: Top 10 Most Underpaid", '-' * 25)
underpaid_show = underpaid[["employee_id", "role_clean", "company_type_clean", "years_exp",
                            "ctc_lpa", "group_median", "gap"]].copy()
underpaid_show[["ctc_lpa", "group_median", "gap"]] = underpaid_show[["ctc_lpa", "group_median", "gap"]].round(1)
print(underpaid_show.to_string(index=False))

------------------------- Q3.5: Top 10 Most Underpaid -------------------------
employee_id       role_clean company_type_clean  years_exp  ctc_lpa  group_median  gap
    BLR0925      SDE Backend                MNC          4     19.4          27.3 -7.9
    BLR0072      SDE Backend                MNC          4     19.5          27.3 -7.8
    BLR0733 Business Analyst                MNC          7     31.1          38.5 -7.4
    BLR0301  Product Manager                MNC          2     24.7          31.4 -6.8
    BLR0453 Business Analyst                MNC          6     32.1          38.5 -6.4
    BLR0611  Product Manager                MNC          2     25.1          31.4 -6.3
    BLR0451     SDE Frontend                MNC          4     19.5          25.5 -6.0
    BLR0203      SDE Backend                MNC          4     21.3          27.3 -6.0
    BLR0033     SDE Frontend                MNC          4     19.6          25.5 -5.9
    BLR0486  Product Manager                MNC   

**Insight (Q3.5):** The most underpaid person is **BLR0925**, an SDE Backend at an
MNC with 4 years earning **19.4 LPA** against a **27.3 LPA** peer median (gap **-7.9 LPA**).
Strikingly, **all 10** of the most-underpaid professionals work at an MNC.

## Task 4 - Build the Printed Report

The LinkedIn screenshot. Uses f-strings with width specifiers and `=` / `-` dividers.
All numbers are pulled from the Task 3 results - nothing is hardcoded.

In [18]:
# Task 4: printed report (uses result, band_ctc, skill_table, company_ctc, underpaid)
W = 60

print("=" * W)
print("  BANGALORE TECH SALARY DECODER")
print("  Built by [G.Vaishanth] | The Unlox Academy | 2-Hour Live Project")
print("=" * W)
print("  Dataset :", len(df), "Bengaluru tech professionals (synthetic)")
print("  Period  : 2024 employment snapshot")
print()

# median CTC by role with a bar chart
print("  ---- MEDIAN CTC BY ROLE (in LPA) ----")
max_median = result["median"].max()
for role in result.index:
    value = result.loc[role, "median"]
    bar = "#" * int(round(value / max_median * 20))
    print(f"  {role:<26}{bar:<20}{value:>5.1f}")
print()

# SDE Backend experience curve
print("  ---- SDE BACKEND CTC BY EXPERIENCE BAND ----")
prev = None
for band in band_ctc.index:
    value = band_ctc[band]
    if prev is None:
        print(f"  {str(band):<5} years   : {value:>5.1f} LPA median")
    else:
        growth = round((value - prev) / prev * 100)
        print(f"  {str(band):<5} years   : {value:>5.1f} LPA median   (+{growth}% growth)")
    prev = value
print()

# skill premium for SDEs
print("  ---- SKILL PREMIUM FOR SDEs (median CTC) ----")
print("  Skill             With Skill    Without   Premium")
for i in range(len(skill_table)):
    sk = skill_table.iloc[i]
    print(f"  {sk['skill']:<16}{sk['with_skill']:>7.1f} LPA   {sk['without']:>6.1f}{sk['premium_pct']:>+7.0f}%")
print()

# company type premium
print("  ---- COMPANY-TYPE PREMIUM (SDE Backend, same role) ----")
unicorn_ctc = company_ctc["Unicorn"]
print(f"  {'Unicorn':<12}{unicorn_ctc:>5.1f} LPA   <- baseline ceiling")
for ct in ["MNC", "Mid-size", "Early-stage"]:
    value = company_ctc[ct]
    diff = round((value - unicorn_ctc) / unicorn_ctc * 100)
    print(f"  {ct:<12}{value:>5.1f} LPA   ({diff}% vs Unicorn)")
print()

# top 5 most underpaid
print("  ---- TOP 5 MOST-UNDERPAID PROFESSIONALS ----")
for i in range(5):
    row = underpaid.iloc[i]
    print(f"  {row['employee_id']:<9}{row['role_clean']:<18}{row['company_type_clean']:<10}"
          f"{int(row['years_exp'])} yrs   gap: {round(row['gap'], 1):+.1f} LPA")
print()

# key insights (numbers pulled from the analysis, not typed by hand)
top_skill = skill_table.iloc[0]
worst = underpaid.iloc[0]
print("  KEY INSIGHTS")
print(f"  1. {top_skill['skill']} adds +{top_skill['premium_pct']:.1f}% to SDE median CTC "
      f"({top_skill['with_skill']} vs {top_skill['without']} LPA).")
print(f"  2. Unicorns pay {premium:.1f}% more than MNCs for SDE Backend "
      f"({company_ctc['Unicorn']:.1f} vs {company_ctc['MNC']:.1f} LPA).")
print(f"  3. All top 10 underpaid pros are at MNCs; worst is {worst['employee_id']} "
      f"({worst['gap']:+.1f} LPA gap).")
print("=" * W)

  BANGALORE TECH SALARY DECODER
  Built by [G.Vaishanth] | The Unlox Academy | 2-Hour Live Project
  Dataset : 1000 Bengaluru tech professionals (synthetic)
  Period  : 2024 employment snapshot

  ---- MEDIAN CTC BY ROLE (in LPA) ----
  Product Manager           #################### 31.3
  ML Engineer               ##################   27.5
  Data Scientist            ###############      24.1
  Site Reliability Engineer ###############      23.3
  SDE Full-Stack            ##############       22.4
  Analytics Engineer        ##############       21.4
  SDE Frontend              ##############       21.2
  SDE Backend               #############        21.1
  Business Analyst          #############        19.9
  DevOps Engineer           ############         19.4
  UI/UX Designer            ############         18.9
  BI Analyst                ###########          16.5
  Data Analyst              ##########           16.3

  ---- SDE BACKEND CTC BY EXPERIENCE BAND ----
  0-1   years  

## Task 5 - Three Insights

Each is data-specific, cites a number, and implies an action.

### Insight 1 - One skill is worth a promotion's worth of pay
For SDEs, listing **System Design** is tied to a **25.3 LPA** median versus **21.0 LPA**
without it - a **+20.5% (~4.3 LPA)** premium, the largest of any skill tested.
**Action:** Spend a few weeks on system design before interview season. It lifts pay
more than adding any single tool or cloud certification.

### Insight 2 - Switching employer type beats switching skills
For the **identical SDE Backend role**, Unicorns pay a **27.4 LPA** median vs **20.5 LPA
at MNCs (+33.7%)** and **18.6 LPA at Early-stage (+47%)**.
**Action:** Weight unicorn-stage companies heavily in your job search and use the 27.4 LPA
unicorn median as your negotiation anchor, not the lower MNC band.

### Insight 3 - Mid-level MNC engineers are the most underpaid cohort
**All 10** of the most-underpaid professionals work at an **MNC**. The worst case,
**BLR0925**, is an SDE Backend with 4 years earning **19.4 LPA** against a **27.3 LPA**
peer median - a **-7.9 LPA** gap.
**Action:** If you are a 3-5 year engineer at an MNC sitting below ~25 LPA, get one
external offer to benchmark and renegotiate.

## Task 6 - Code Review (fresh-eyes pass)

**Checklist**
- [x] Every section starts with a markdown title
- [x] Every code cell has comments explaining what it does
- [x] Variable names are clear (df, result, band_ctc, skill_table, company_ctc, underpaid)
- [x] Notebook runs top to bottom without errors
- [x] Only pandas and numpy used;
- [x] The printed report is the last output cell

**AI-assistance disclosure**
- The CTC parser (`convert_ctc`) and the role-name mapping were built with AI help,
  then checked against the actual data.
- Every number in the report and insights comes from running the code, not from guessing.

In [19]:
# quick self-check that the cleaned data looks correct
print('-' * 25, "Self Check", '-' * 25)
print("Rows :", len(df))
print("Columns :", len(df.columns))
print("ctc_lpa present :", "ctc_lpa" in df.columns)
print("Number of clean roles :", df["role_clean"].nunique())
print("Company types :", list(df["company_type_clean"].unique()))
print("Education tiers :", list(df["education_tier_clean"].unique()))
print("Duplicate employee_ids :", df["employee_id"].duplicated().sum())
print("CTC range :", round(df["ctc_lpa"].min(), 1), "to", round(df["ctc_lpa"].max(), 1), "LPA")
print("Any role not mapped :", df["role_clean"].isna().sum())

------------------------- Self Check -------------------------
Rows : 1000
Columns : 23
ctc_lpa present : True
Number of clean roles : 13
Company types : ['Unicorn', 'MNC', 'Mid-size', 'Early-stage']
Education tiers : ['Tier 1', 'Tier 2', 'Tier 3']
Duplicate employee_ids : 0
CTC range : 5.2 to 84.4 LPA
Any role not mapped : 0
